# Nool — Enterprise Knowledge Assistant: Company Search Index

This notebook builds the company document search files with a Colab GPU. It uses the 45 fictional company documents already in the repository. It does not change the 50,000-passage benchmark.

1. In Colab, select **Runtime → Change runtime type → T4 GPU**.
2. Run the cells below from top to bottom.
3. Download `company_search_artifact.zip` from `MyDrive/Nool/` when the last cell finishes.

The ZIP contains search indexes, not a trained model. The notebook also prints retrieval checks. These checks measure source finding, not answer quality.


## Set up Colab

This cell checks the GPU, connects Drive, gets the latest project code, and installs the search dependencies. Approve the Google Drive prompt when Colab asks.


In [ ]:
import subprocess
import sys
from pathlib import Path

import torch
from google.colab import drive

if not torch.cuda.is_available():
    raise RuntimeError("Select a GPU runtime in Colab, then run this cell again.")

drive.mount("/content/drive")
repo = Path("/content/Nool")
if repo.exists():
    subprocess.run(["git", "-C", str(repo), "pull", "--ff-only"], check=True)
else:
    subprocess.run(
        ["git", "clone", "https://github.com/nayanojwal0810/Nool.git", str(repo)],
        check=True,
    )

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "pyyaml>=6.0.1",
        "bm25s==0.3.16",
        "PyStemmer==3.1.0",
        "pypdf>=5.0,<7.0",
        "transformers==5.22.1",
    ],
    check=True,
)
sys.path.insert(0, str(repo / "src"))
commit = subprocess.check_output(
    ["git", "-C", str(repo), "rev-parse", "HEAD"], text=True
).strip()
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Project commit: {commit}")

## Build both search indexes

The first command reads the company files into a temporary database. The second creates BM25 and Contriever indexes from the same current passages. The model download may take a few minutes. Keep this Colab tab open until the cell finishes.

For a document update, first run `scripts/sync_demo_documents.py` on your computer. Upload `data/processed/demo_documents.sqlite` to `MyDrive/Nool/input/company_documents.sqlite`, then set `USE_UPDATED_DATABASE = True` in the next cell. This keeps the document versions aligned with your computer. For the original collection, leave it `False`.


In [ ]:
import os
import shutil

USE_UPDATED_DATABASE = False
database = repo / "data/processed/demo_documents.sqlite"
index = repo / "indexes/company"
project_env = os.environ.copy()
project_env["PYTHONPATH"] = str(repo / "src")
if USE_UPDATED_DATABASE:
    uploaded = Path("/content/drive/MyDrive/Nool/input/company_documents.sqlite")
    if not uploaded.is_file():
        raise FileNotFoundError(f"Upload the updated database to {uploaded}")
    database.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(uploaded, database)
else:
    subprocess.run(
        [
            sys.executable,
            "scripts/ingest_demo_documents.py",
            "--database",
            str(database),
        ],
        cwd=repo,
        env=project_env,
        check=True,
    )
subprocess.run(
    [
        sys.executable,
        "scripts/build_company_search.py",
        "--database",
        str(database),
        "--output",
        str(index),
        "--device",
        "cuda",
    ],
    cwd=repo,
    env=project_env,
    check=True,
)

## Check source retrieval and access

The first group asks 150 questions that have known source documents. A case passes when all of its listed sources appear in the first five distinct documents. The second group checks seven employee questions whose only target document is restricted. A zero result here means none of those restricted documents appeared. These are search checks only; they do not judge generated answers.

This question set describes the original documents. The cell skips these scores when `USE_UPDATED_DATABASE` is `True`; the update checks belong in the local test suite.


In [ ]:
import json
from collections import Counter

from nool.data.documents import DocumentStore
from nool.retrieval.bm25 import BM25Retriever
from nool.retrieval.company import CompanyHybridSearch, corpus_fingerprint
from nool.retrieval.contriever import ContrieverEncoder, ContrieverRetriever

store = DocumentStore(database)
bm25 = BM25Retriever.load(index / "bm25")
dense = ContrieverRetriever.load(
    index / "contriever",
    index / "corpus.jsonl",
    encoder=ContrieverEncoder(device="cuda"),
)
fingerprint = (index / "source_fingerprint.txt").read_text(encoding="utf-8")
assert fingerprint == corpus_fingerprint(store), "The index is outdated. Rebuild it."
search = CompanyHybridSearch(store, bm25, dense, indexed_fingerprint=fingerprint)
questions = [
    json.loads(line)
    for line in (repo / "data/demo/question_cases.jsonl")
    .read_text(encoding="utf-8")
    .splitlines()
]

supported = {"direct", "paraphrase", "multiple_sources", "near_miss"}
checked = Counter()
passed = Counter()
failures = []
for case in questions if not USE_UPDATED_DATABASE else []:
    if case["scenario"] not in supported:
        continue
    found = {
        hit.document_id
        for hit in search.search(case["question"], case["user_id"], top_k=5)
    }
    expected = set(case["source_document_ids"])
    checked[case["scenario"]] += 1
    if expected <= found:
        passed[case["scenario"]] += 1
    else:
        failures.append(case["case_id"])

denied_failures = []
for case in questions if not USE_UPDATED_DATABASE else []:
    if case["scenario"] != "access_denied":
        continue
    forbidden = case["state_change"]["document_id"]
    found = {
        hit.document_id
        for hit in search.search(case["question"], case["user_id"], top_k=5)
    }
    if forbidden in found:
        denied_failures.append(case["case_id"])

if USE_UPDATED_DATABASE:
    print("Baseline question scores skipped for updated documents.")
else:
    for scenario in sorted(supported):
        print(
            f"{scenario}: {passed[scenario]}/{checked[scenario]} sources found in top 5"
        )
    print(f"Restricted source leaks: {len(denied_failures)}/7")
    print(f"Retrieval case IDs to review: {failures}")
    print(f"Access case IDs to review: {denied_failures}")
if denied_failures:
    raise RuntimeError(
        "Restricted documents appeared in search results. Do not use this artifact."
    )

## Save the files to Google Drive

This cell packages the complete company index into one ZIP. It prints a checksum so the download can be checked on your computer.


In [ ]:
import hashlib
import zipfile

destination = Path("/content/drive/MyDrive/Nool/company_search_artifact.zip")
destination.parent.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(destination, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for file in sorted(index.rglob("*")):
        if file.is_file():
            archive.write(file, file.relative_to(index.parent))

digest = hashlib.sha256(destination.read_bytes()).hexdigest()
print(f"Saved: {destination}")
print(f"Size: {destination.stat().st_size / 1_000_000:.1f} MB")
print(f"SHA-256: {digest}")

## Bring the ZIP back to Nool

Download `company_search_artifact.zip` from Google Drive. Put it in the project folder on your computer, then extract it into `indexes/`. The result should include `indexes/company/corpus.jsonl`, `indexes/company/bm25/`, and `indexes/company/contriever/`.

Run this from the project folder in PowerShell:

```powershell
Expand-Archive -Path .\company_search_artifact.zip -DestinationPath .\indexes -Force
.\.venv\Scripts\python scripts/search_company.py "How many annual leave days do I get?" --user employee_demo
```

The indexes are excluded from Git. Share the printed retrieval counts and failed case IDs for review. If the notebook reports an error, share the error text and the cell heading.

For an updated collection, keep the local SQLite database you uploaded. Extract the new ZIP into `indexes/` to replace the outdated search files, then run a local search.
